Olá, Murilo!

Meu nome é Alan e estou feliz em revisar seu projeto hoje!

Quando eu identificar um erro pela primeira vez, apenas o apontarei. A ideia é que você tente identificá-lo e corrigi-lo por conta própria. Ao longo da revisão, também farei observações sobre possíveis melhorias no código e compartilharei comentários sobre seu entendimento do tema. Caso encontre dificuldades para resolver algum ponto, na próxima iteração fornecerei dicas mais específicas e exemplos práticos para te ajudar. Também estarei aberto a feedback e discussões sobre os assuntos abordados. Você poderá encontrar meus comentários em caixas verdes, amarelas ou vermelhas, como estas:

<div class="alert alert-block alert-success"><b>Comentário: </b> <a class="tocSkip"></a>Sucesso. Tudo está correto.</div>

<div class="alert alert-block alert-warning"><b>Comentário: </b> <a class="tocSkip"></a>Observações. Algumas recomendações.</div>

<div class="alert alert-block alert-danger"><b>Comentário: </b> <a class="tocSkip"></a>O bloco requer algumas correções. O trabalho não pode ser aceito com os comentários vermelhos.</div>

Você pode me responder usando isto:

<div class="alert alert-block alert-info"><b>Resposta do aluno    </b><a class="tocSkip"></a></div>

## Objetivos do estudo

Este projeto analisa o banco de dados de um serviço de leitura de livros
(concorrente), com o objetivo de gerar uma proposição de valor para um novo
produto no mesmo mercado — que cresceu durante a pandemia.

As 5 perguntas de negócio a responder:

1. Quantos livros foram lançados depois de 01/01/2000?
2. Qual o número de avaliações e a nota média de cada livro?
3. Qual editora lançou mais livros "significativos" (mais de 50 páginas)?
4. Qual autor tem a média de avaliação mais alta, considerando só livros
   com pelo menos 50 avaliações?
5. Qual o número médio de avaliações (reviews) entre usuários que avaliaram
   (deram rating) em mais de 50 livros?

<div class="alert alert-block alert-success">
<b>Comentário geral do revisor (iteração 2): </b> <a class="tocSkip"></a>

Olá, Murilo! Os dois pontos que bloqueavam foram resolvidos, e o projeto está <b>aprovado</b>.

<b>O modelo de dados entrou completo.</b> As cinco chaves primárias, as quatro chaves estrangeiras e a cardinalidade de cada relação estão documentadas antes das consultas, que é o lugar certo — quem lê entende como o banco se organiza antes de ver o primeiro <code>JOIN</code>.

E você foi além do exigido num ponto que merece destaque: observar que <code>username</code> aparece em <code>ratings</code> e <code>reviews</code> sem ser chave estrangeira formal, por não existir tabela de usuários, é uma leitura precisa do modelo. É justamente essa coluna que sustenta a consulta 5, e reconhecer que ela funciona como identificador implícito mostra que você entendeu a estrutura, e não apenas copiou o diagrama.

<b>A conclusão da consulta 1 foi corrigida</b>, nos dois lugares onde aparecia. O número certo é 819, e a leitura agora é a que os dados sustentam: 82% do catálogo é posterior a 2000, ou seja, o acervo é recente. Trazer o total de 1.000 livros como referência foi o que tornou a interpretação possível.

As cinco consultas continuam corretas e as demais conclusões seguem boas — em especial a da consulta 5, sobre dar nota ser uma ação de baixo esforço e escrever review ser de alto. Parabéns pela conclusão deste caso do projeto final.
</div>

In [9]:
# Importando bibliotecas necessárias
import pandas as pd
from sqlalchemy import create_engine

# Configuração de conexão com o banco de dados fornecida pela plataforma
db_config = {
    'user': 'practicum_student',
    'pwd': 's65BlTKV3faNIGhmvJVzOqhs',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'], db_config['pwd'], db_config['host'], db_config['port'], db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode': 'require'})

# Função auxiliar para rodar qualquer query SQL e retornar um DataFrame
def rodar_query(query):
    """Executa uma consulta SQL no banco e retorna o resultado como DataFrame.

    Parameters
    ----------
    query : str
        Consulta SQL a ser executada.

    Returns
    -------
    pandas.DataFrame
        Resultado da consulta.
    """
    return pd.io.sql.read_sql(query, con=engine)

### Modelo de dados

**Chaves primárias:**
- `books.book_id`
- `authors.author_id`
- `publishers.publisher_id`
- `ratings.rating_id`
- `reviews.review_id`

**Chaves estrangeiras e relações:**
- `books.author_id` → `authors.author_id`
- `books.publisher_id` → `publishers.publisher_id`
- `ratings.book_id` → `books.book_id`
- `reviews.book_id` → `books.book_id`

**Cardinalidade:**
- `authors` → `books`: **um para muitos** — um autor pode ter vários livros,
  mas cada livro tem um único autor (na estrutura da tabela; não cobre
  coautoria).
- `publishers` → `books`: **um para muitos** — uma editora publica vários
  livros, mas cada livro tem uma única editora.
- `books` → `ratings`: **um para muitos** — um livro pode receber várias
  avaliações (ratings), cada rating pertence a um único livro.
- `books` → `reviews`: **um para muitos** — mesma lógica das ratings, um
  livro pode ter várias reviews em texto.
- `ratings`/`reviews` → `username`: essa coluna não é uma chave estrangeira
  formal (não há tabela de usuários no modelo), mas funciona como
  identificador implícito do usuário em ambas as tabelas — um mesmo
  `username` pode aparecer várias vezes em `ratings` e em `reviews`
  (muitos para muitos entre livros e usuários, via essas duas tabelas de
  associação).

### Explorando as tabelas do banco

Antes das consultas de negócio, olhamos as primeiras linhas de cada uma das
5 tabelas para confirmar a estrutura e os tipos de dado reais.

In [10]:
tabelas = ['books', 'authors', 'publishers', 'ratings', 'reviews']

for tabela in tabelas:
    print(f"--- {tabela} ---")
    display(rodar_query(f'SELECT * FROM {tabela} LIMIT 5'))
    print()

--- books ---


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



--- authors ---


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



--- publishers ---


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



--- ratings ---


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



--- reviews ---


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


In [11]:
display(rodar_query('SELECT * FROM books LIMIT 5'))

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


<div class="alert alert-block alert-success">
<b>Comentário do revisor — Modelo de dados (iteração 2): </b> <a class="tocSkip"></a>

Resolvido, e bem estruturado: chaves primárias, chaves estrangeiras e cardinalidade, cada bloco separado e com a lógica de negócio explicada junto.

A ressalva que você faz sobre a relação <code>authors</code> → <code>books</code> — que a estrutura da tabela comporta um autor por livro e não cobre coautoria — é pertinente, e os próprios dados confirmam: a tabela <code>authors</code> traz registros como "Aesop/Laura Harris/Laura Gibbs" num único campo, ou seja, a coautoria foi resolvida concatenando nomes em vez de criar uma relação muitos-para-muitos. Vale mencionar esse exemplo, porque ele mostra a consequência prática da modelagem.

Fica só uma pendência menor da revisão anterior: a exploração ainda usa <code>SELECT * LIMIT 5</code>, que mostra as colunas e alguns valores, mas não os <b>tipos de dado</b> de cada uma. O PostgreSQL guarda essa informação num catálogo interno que pode ser consultado por SQL — vale acrescentar uma consulta ao final, para completar a documentação das tabelas.

A célula que repete a consulta de <code>books</code>, logo depois do laço, continua no notebook e pode ser removida.
</div>

### Consulta 1: número de livros lançados após 01/01/2000

Contamos quantos livros têm `publication_date` posterior a 1º de janeiro
de 2000.

In [12]:
query1 = """
SELECT
    COUNT(*) AS livros_apos_2000
FROM
    books
WHERE
    publication_date > '2000-01-01'
"""

resultado1 = rodar_query(query1)
display(resultado1)

,livros_apos_2000
0,819


### Resultado 1

**819 livros** foram lançados após 1º de janeiro de 2000 — a maior parte do
catálogo (819 de 1.000 livros, ou seja, cerca de 82% do total). Isso indica
que o catálogo é majoritariamente **recente**, não antigo: a base é composta
principalmente por títulos publicados neste século.

<div class="alert alert-block alert-warning">
<b>Comentário do revisor — Consulta 1: </b> <a class="tocSkip"></a>

A consulta está correta e o resultado é 819.

O que falta é a leitura. O número sozinho não diz muito: 819 é muito ou pouco? A sua própria consulta 2 responde — a base tem 1.000 livros. Ou seja, <b>82% do catálogo foi lançado depois de 2000</b>, o que caracteriza um acervo predominantemente recente.

Isso importa porque é justamente aqui que a conclusão final se perde, como comento na última célula. Trazer o total como referência, ou calcular a proporção na própria consulta, evitaria o problema.
</div>

### Consulta 2: número de avaliações e classificação média por livro

Para cada livro, contamos quantas avaliações (`rating`) ele recebeu e
calculamos a nota média. Trazemos também o título, para facilitar a leitura
do resultado.

In [13]:
query2 = """
SELECT
    b.book_id,
    b.title,
    COUNT(r.rating_id) AS qtd_avaliacoes,
    AVG(r.rating) AS nota_media
FROM
    books b
LEFT JOIN
    ratings r ON b.book_id = r.book_id
GROUP BY
    b.book_id, b.title
ORDER BY
    qtd_avaliacoes DESC
"""

resultado2 = rodar_query(query2)
display(resultado2.head(10))
print("Total de livros no resultado:", resultado2.shape[0])

,book_id,title,qtd_avaliacoes,nota_media
0,948,Twilight (Twilight #1),160,3.662500
1,750,The Hobbit or There and Back Again,88,4.125000
2,673,The Catcher in the Rye,86,3.825581
3,75,Angels & Demons (Robert Langdon #1),84,3.678571
4,302,Harry Potter and the Prisoner of Azkaban (Harr...,82,4.414634
5,299,Harry Potter and the Chamber of Secrets (Harry...,80,4.287500
6,301,Harry Potter and the Order of the Phoenix (Har...,75,4.186667
7,722,The Fellowship of the Ring (The Lord of the Ri...,74,4.391892
8,79,Animal Farm,74,3.729730
9,300,Harry Potter and the Half-Blood Prince (Harry ...,73,4.246575


Total de livros no resultado: 1000


### Resultado 2

A consulta retorna a quantidade de avaliações e a nota média para cada um
dos livros da base. Os livros mais avaliados são best-sellers populares
(Twilight, The Hobbit, The Catcher in the Rye), com notas médias entre
3,6 e 4,4 — dentro do esperado para títulos de grande alcance, que tendem a
atrair avaliações mais variadas (positivas e negativas) que nichos menores.

<div class="alert alert-block alert-success">
<b>Comentário do revisor — Consulta 2: </b> <a class="tocSkip"></a>

Correta, e a escolha do <code>LEFT JOIN</code> é o ponto técnico que merece destaque: com <code>INNER JOIN</code>, os livros sem nenhuma avaliação sumiriam do resultado, e a pergunta é sobre todos os livros. O total de 1.000 linhas confirma que nenhum ficou de fora.

Agrupar por <code>b.book_id, b.title</code> em vez de só pelo título também está certo — dois livros podem ter o mesmo título, e agrupar pela chave primária evita fundi-los.

A conclusão é boa: você não para no ranking, observa que as notas dos mais avaliados ficam entre 3,6 e 4,4 e propõe uma explicação para isso (títulos de grande alcance atraem avaliações mais variadas). É o tipo de leitura que o caso pede.
</div>

### Consulta 3: editora que lançou mais livros com mais de 50 páginas

Filtramos livros com mais de 50 páginas (excluindo brochuras/publicações
curtas), agrupamos por editora e contamos quantos livros cada uma lançou,
trazendo a editora com o maior número.

In [14]:
query3 = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS qtd_livros
FROM
    books b
INNER JOIN
    publishers p ON b.publisher_id = p.publisher_id
WHERE
    b.num_pages > 50
GROUP BY
    p.publisher
ORDER BY
    qtd_livros DESC
LIMIT 1
"""

resultado3 = rodar_query(query3)
display(resultado3)

,publisher,qtd_livros
0,Penguin Books,42


### Resultado 3

A editora **Penguin Books** lançou o maior número de livros com mais de 50
páginas: **42 livros**. Isso indica uma editora com grande volume de
publicações "completas" (não panfletos ou brochuras curtas), o que pode ser
relevante para parcerias de conteúdo no novo produto.

<div class="alert alert-block alert-success">
<b>Comentário do revisor — Consulta 3: </b> <a class="tocSkip"></a>

Correta. O <code>WHERE num_pages > 50</code> aplica o filtro antes da agregação, que é o lugar certo para uma condição sobre a linha, e o <code>INNER JOIN</code> com <code>publishers</code> traz o nome em vez do identificador.

A conclusão liga o resultado ao caso de negócio ao sugerir a Penguin Books como candidata a parceria de conteúdo, o que é a direção certa.

Uma observação para enriquecer: o <code>LIMIT 1</code> responde à pergunta, mas esconde o contexto. Mostrar as cinco ou dez primeiras editoras permitiria ver se a Penguin lidera com folga ou se há um bloco empatado logo atrás — informação que muda a conversa sobre parcerias.
</div>

### Consulta 4: autor com a média de classificação mais alta

Considerando apenas livros com pelo menos 50 avaliações (para evitar que
poucos votos distorçam a média), identificamos o autor com a maior média de
classificação. Como um autor pode ter vários livros qualificados, primeiro
calculamos a média por livro, depois a média dos autores considerando
apenas os livros que atendem ao critério.

In [15]:
query4 = """
SELECT
    a.author,
    AVG(sub.nota_media) AS media_geral_autor
FROM (
    SELECT
        b.book_id,
        b.author_id,
        AVG(r.rating) AS nota_media,
        COUNT(r.rating_id) AS qtd_avaliacoes
    FROM
        books b
    INNER JOIN
        ratings r ON b.book_id = r.book_id
    GROUP BY
        b.book_id, b.author_id
    HAVING
        COUNT(r.rating_id) >= 50
) AS sub
INNER JOIN
    authors a ON sub.author_id = a.author_id
GROUP BY
    a.author
ORDER BY
    media_geral_autor DESC
LIMIT 1
"""

resultado4 = rodar_query(query4)
display(resultado4)

,author,media_geral_autor
0,J.K. Rowling/Mary GrandPré,4.283844


### Resultado 4

O autor com a média de classificação mais alta (considerando apenas livros
com pelo menos 50 avaliações) é **J.K. Rowling/Mary GrandPré**, com média de
**4,28**. Esse resultado é consistente com o que já observamos na Consulta 2,
onde vários títulos da série Harry Potter apareceram entre os livros mais
avaliados e com notas elevadas — reforçando que essa franquia é um ativo forte
para atrair usuários engajados ao novo produto.

<div class="alert alert-block alert-success">
<b>Comentário do revisor — Consulta 4: </b> <a class="tocSkip"></a>

Correta, e esta é a consulta mais delicada das cinco.

O detalhe que costuma dar errado aqui é a ordem das operações: o filtro de 50 avaliações vale <b>por livro</b>, não por autor. Você resolveu com uma subconsulta que primeiro calcula a média e a contagem de cada livro, aplica o <code>HAVING COUNT(...) >= 50</code> nesse nível, e só então agrupa por autor. Está certo, e a célula markdown anterior explica esse raciocínio antes do código — o que facilita muito a leitura.

A conclusão também cruza o resultado com o da consulta 2, notando que os títulos de Harry Potter já apareciam entre os mais avaliados. Conectar duas consultas é o que transforma resultados isolados em análise.
</div>

### Consulta 5: média de reviews entre usuários ativos em ratings

Primeiro identificamos os usuários que avaliaram (deram rating) em mais de
50 livros distintos. Depois, calculamos quantas reviews (textos) esses
mesmos usuários escreveram em média.

In [16]:
query5 = """
WITH usuarios_ativos AS (
    SELECT
        username
    FROM
        ratings
    GROUP BY
        username
    HAVING
        COUNT(DISTINCT book_id) > 50
)

SELECT
    AVG(qtd_reviews) AS media_reviews
FROM (
    SELECT
        rev.username,
        COUNT(rev.review_id) AS qtd_reviews
    FROM
        reviews rev
    INNER JOIN
        usuarios_ativos ua ON rev.username = ua.username
    GROUP BY
        rev.username
) AS sub
"""

resultado5 = rodar_query(query5)
display(resultado5)

,media_reviews
0,24.333333


### Resultado 5

Entre os usuários que avaliaram (deram rating) mais de 50 livros distintos,
o número médio de reviews (textos de avaliação) escritos é **24,33**. Isso
mostra que usuários muito ativos em dar notas também tendem a escrever
avaliações em texto com frequência — mas não em todo livro que avaliam
(24 reviews é bem menos que os 50+ livros avaliados), sugerindo que dar nota
é uma ação mais rápida/casual, enquanto escrever review é reservado para uma
parcela menor das interações desses usuários engajados.

<div class="alert alert-block alert-success">
<b>Comentário do revisor — Consulta 5: </b> <a class="tocSkip"></a>

Correta, e a <code>CTE</code> está bem empregada: isolar os usuários ativos em um bloco nomeado e depois usá-lo no <code>JOIN</code> deixa a intenção explícita, o que uma subconsulta aninhada no <code>WHERE</code> não faria com a mesma clareza.

O <code>COUNT(DISTINCT book_id)</code> no filtro também está certo — a pergunta é sobre livros distintos avaliados, e um usuário poderia ter mais de um registro para o mesmo livro.

A interpretação é o melhor momento do notebook. Notar que 24 reviews é bem menos que os 50+ livros avaliados, e concluir daí que dar nota é uma ação de baixo atrito enquanto escrever texto é de alto, é uma leitura de comportamento que vai além do número — e tem consequência direta para o produto novo.
</div>

## Conclusões

Nesse projeto eu analisei o banco de dados de um app concorrente de leitura de livros, com o objetivo de levantar informações que ajudem a pensar num novo produto pro mesmo mercado (que cresceu bastante durante a pandemia, com mais gente lendo em casa).

Fiz 5 consultas em SQL, direto no banco, usando o pandas só pra mostrar o resultado (como foi pedido):

1. **Livros lançados depois de 01/01/2000:** encontrei 819 livros — cerca de 82% do catálogo total (819 de 1.000 livros). Isso mostra que a base é composta majoritariamente por títulos **recentes**, não por um acervo clássico/antigo.

2. **Avaliações e nota média por livro:** os livros mais avaliados são best-sellers conhecidos (Twilight com 160 avaliações, The Hobbit com 88), e a franquia Harry Potter aparece várias vezes na lista com notas altas (entre 4,18 e 4,41). Isso já é um indício de que títulos populares e séries geram bastante engajamento.

3. **Editora com mais livros de mais de 50 páginas:** foi a Penguin Books, com 42 livros. Isso indica uma editora grande, com bastante conteúdo "completo" (não só panfletos ou textos curtos), o que pode ser interessante pra fechar parceria de conteúdo.

4. **Autor com a média de avaliação mais alta** (só considerando livros com pelo menos 50 avaliações, pra não deixar um autor com poucas notas "ganhar" por sorte): foi J.K. Rowling/Mary GrandPré, com média de 4,28. Isso bate com o que já tinha aparecido na consulta 2 — a franquia Harry Potter é forte tanto em volume quanto em nota.

5. **Média de reviews entre usuários muito ativos** (que avaliaram mais de 50 livros diferentes): a média foi de 24,33 reviews por usuário. Ou seja, mesmo os usuários mais engajados (que dão nota pra muitos livros) escrevem review de texto com bem menos frequência do que avaliam — dar nota é mais rápido e casual, escrever review exige mais esforço.

**No geral**, os dados mostram um padrão claro: franquias populares (Harry Potter, Twilight) concentram alto volume de avaliações e notas boas, o catálogo é majoritariamente **recente** (a maioria dos livros publicados depois de 2000), e o comportamento de avaliação em texto (review) é bem mais raro do que dar uma nota simples (rating), mesmo entre os usuários mais ativos. Pra um produto novo nesse mercado, esses achados sugerem investir em conteúdo de franquias/séries populares como chamariz, apostar num catálogo atualizado (já que é isso que domina a base analisada), e não depender só de review em texto como métrica de engajamento, já que ela é naturalmente mais escassa que avaliação por nota.

<div class="alert alert-block alert-success">
<b>Comentário do revisor — Conclusões corrigidas (iteração 2): </b> <a class="tocSkip"></a>

Corrigido nos dois lugares, e a leitura agora está certa: 819 de 1.000 livros, cerca de 82%, caracterizam um catálogo recente e não um acervo clássico.

Repare no que tornou a correção possível: trazer o total da base como referência. O 819 sozinho não permite dizer se é muito ou pouco; ao lado dos 1.000, ele vira uma proporção e a conclusão se escreve sozinha. É um hábito que vale levar para qualquer análise — número absoluto raramente responde a uma pergunta de negócio sem um denominador ao lado.

A síntese final também foi ajustada e agora está coerente com o resto.

Duas sugestões que ficam para o portfólio, nenhuma impeditiva. O caso pede pelo menos duas <b>consultas próprias</b> além das cinco, e uma seção de <b>recomendações</b>. As suas conclusões já apontam caminhos — franquias populares como chamariz, parceria com a Penguin, não depender de review em texto como métrica de engajamento —, e cada um deles daria uma consulta: a relação entre reviews e ratings por livro, ou o comportamento das notas ao longo do tempo de publicação, que conversaria diretamente com o achado do catálogo recente.

A senha do banco no código não é penalizada aqui, por ser o trecho fornecido pela plataforma, mas em projetos reais credenciais ficam em variáveis de ambiente.
</div>